## 선택 학습 시작

① 맨 위 시작 셀 실행 → ② 문제 풀기 → ③ 맨 아래 종료 셀 실행 후 종료 입력 → ④ Ctrl+S로 노트북 저장 → ⑤ 이 대화에 다 풀었어라고 알려주기.

공통 \_shared 타이머를 사용해요. 기록은 이 교재의 .study_timer에만 남아요. KST는 한국 표준시이며 UTC(협정 세계시)보다 9시간 빨라요. 경과 시간은 쉬거나 노트북을 닫은 시간도 포함하는 벽시계 시간이에요. 종료 시각은 종료 셀에서 확인한 시각이며 저장·채팅 전송 시각과 달라요. 모두 실행 중에는 종료를 입력하지 마세요. 타이머는 선택이며 쓰지 않아도 문제를 풀 수 있어요.


In [ ]:
import stat as _stat
from pathlib import Path as _Path


def _load_shared_timer():
    # 교재 ID·날짜·노트북 이름은 제작 때 고정한 값이에요.
    _lesson_id = "day24-sensor-log-analysis"
    _lesson_date = "2026-11-12"
    _notebook_name = "day24.ipynb"

    def _safe(path, directory=False):
        if ".." in path.parts:
            raise RuntimeError("상위 폴더 경로는 사용하지 않아요.")
        for item in [*reversed(path.parents), path]:
            info = item.lstat()
            if (
                _stat.S_ISLNK(info.st_mode)
                or getattr(info, "st_file_attributes", 0) & 0x400
            ):
                raise RuntimeError("바로가기·junction·reparse 경로는 사용하지 않아요.")
        if directory and not _stat.S_ISDIR(info.st_mode):
            raise RuntimeError("예상한 폴더가 아니에요.")
        if not directory and not _stat.S_ISREG(info.st_mode):
            raise RuntimeError("예상한 일반 파일이 아니에요.")
        return path

    _cwd = _safe(_Path.cwd(), directory=True)
    _candidates = []
    for _folder in (_cwd, _cwd / _lesson_id):
        if _folder.name == _lesson_id and (_folder / _notebook_name).is_file():
            _safe(_folder, directory=True)
            _candidates.append(_safe(_folder / _notebook_name))
    if len(_candidates) != 1:
        raise RuntimeError("교재 폴더가 없거나 모호해요.")
    _notebook_path = _candidates[0]
    _shared_dir = _safe(_notebook_path.parent.parent / "_shared", directory=True)
    _helper = _safe(_shared_dir / "study_timer.py")
    # 검사한 파일을 메모리에서 로드해 __pycache__를 만들지 않아요.
    _before = _helper.lstat()
    _source = _helper.read_bytes()
    _after = _safe(_helper).lstat()
    _identity = lambda info: (info.st_dev, info.st_ino, info.st_mtime_ns, info.st_size)
    if _identity(_before) != _identity(_after):
        raise RuntimeError("읽는 동안 공통 도구 파일이 바뀌었어요.")
    _module = {"__name__": "physical_ai_shared_timer"}
    exec(compile(_source, str(_helper), "exec"), _module)
    return _module["create_timer"](
        _lesson_id, _lesson_date, _notebook_name, notebook_path=_notebook_path
    )


try:
    _study_timer = _load_shared_timer()
except (OSError, RuntimeError, SyntaxError):
    print(
        "타이머 경로를 확인하지 못했어요. 루트 또는 교재 폴더에서 열고 _shared 폴더를 함께 보관해 주세요. 기록은 바꾸지 않았어요."
    )
else:
    _study_timer.action("start")

# Day24 센서 기록을 정리하고 다시 분석하기


2026-11-12 · 실제 실습 약 60분을 잠정 목표로 해요.

**오늘 만들 것**

시각과 센서 열 이름이 섞여 들어온 거리 기록을 정리하고 위험 표시를 분석하는 analyze_sensor_log 함수 하나예요. 이 함수를 같은 기록의 기준 변경·열 이름 오류·측정 품질 문제에 적용해요. 필요한 입력은 한 준비 셀에 모았어요.


**이미 확인한 것**

Day23의 보정·판단 연결, 실행 조건 비교, 오래된 관측 수정과 결과 파일 재읽기는 통과했어요. 이 기능들을 다시 구현하거나 이전 교재에서 코드를 가져오지 않아요.

**오늘 연결할 것**

배열의 값과 그 값의 뜻을 함께 관리해요. 앞쪽 센서인지, 언제 측정한 값인지가 맞아야 분석도 맞아요.

**이후 단계**

센서 기록 분석을 실행 가능한 로봇 프로그램에 연결하는 기반을 다져요. Ubuntu/Git 협업·행렬과 3차원·ROS2를 완료한 것으로 표시하지 않아요.


**실습 시간 배분**

- 진입 확인·개념 보충은 필요한 부분만 읽고, 아래 실습 시간과 별도로 사용해요.
- Q1 분석 함수 구현 20분 → Q2 기준 변경 비교 15분.
- Q3 열 이름 오류 디버깅 10분 → Q4 측정 품질 처리 15분.

문제 구현·비교·디버깅·설명에 약 60분을 잠정 배분했어요. 안내를 해석하거나 다른 교재를 찾는 시간으로 채우지 않아요. 이미 익숙한 예제는 건너뛰고 확인 문제로 가도 돼요. 실제 걸린 시간에 따라 다음 분량을 조정해요.


## 짧은 진입 확인과 보충 선택


**선택 경로 · 최대 3분 · 별도 제출 의무 없음**

열 순서가 front, left, right인 거리 배열에서 앞쪽 열만 고르고, 0.40m 이하인 기록을 표시할 수 있나요? 아래 두 행으로 잠깐 시도해요. 앞쪽은 로봇 정면, 왼쪽·오른쪽은 각 방향의 센서예요. 모든 숫자는 이미 보정된 거리 m(미터)예요.

- 직접 작성하고 각 행의 표시를 설명할 수 있으면 **준비 P0**와 **Q1**로 직행해요.
- 열 선택이나 행별 집계가 헷갈리면 다음 보충 예제만 실행한 뒤 Q1로 가요.

미리보기 응답이 없었으므로 배열 숙달로 가정하지 않아요. 이 진입 확인의 결과만으로 오늘 학습을 완료 처리하지 않아요.


In [ ]:
import numpy as np  # NumPy 배열 도구의 관례적인 짧은 이름 np

entry_distances_m = np.array([[0.70, 0.20, 0.80], [0.30, 0.90, 0.60]])
# 열은 앞, 왼쪽, 오른쪽 순서예요. 필요한 줄만 짧게 시도해요.

## 행과 열을 같은 뜻으로 맞추기


**쉬운 뜻**

한 행은 한 시각의 묶음이고, 한 열은 같은 방향 센서의 값이에요.

**생활 비유**

날짜가 섞인 성적표를 날짜순으로 바꿀 때 날짜 칸만 정렬하면 점수가 다른 날에 붙어요. 한 행 전체가 함께 움직여야 해요. 열을 바꿀 때도 과목 이름과 점수를 함께 옮겨야 해요.

**실제 용어**

배열(array), 행(row), 열(column), 열 이름 같은 설명 정보인 메타데이터(metadata)예요. 그림의 행·열은 기록 표의 구조를 나타내요.


**배열 표기와 단위**

- shape는 배열의 형태예요. (N, 3)은 기록 N개와 센서 3개를 뜻해요. N은 개수를 나타내는 기호이며 물리 단위가 없어요.
- 인덱스는 위치 번호이며 0부터 시작해요. 열 0, 1, 2를 뜻 없이 외우지 않고 열 이름과 연결해요.
- t 또는 time은 기록의 가상 시각을 뜻해요. ms는 밀리초(millisecond)이며 1000ms=1s(초)예요. 컴퓨터의 현재 시각이 아니에요.
- distances_m은 이미 보정된 거리 배열이에요. 단위는 m이며 센서 원시 눈금을 다시 보정하지 않아요.


![입력의 시각·값·열 이름을 함께 읽고, 정리한 기록에 같은 기준을 적용해요.](assets/01-analysis-flow.png)

입력의 시각·값·열 이름을 함께 읽고, 정리한 기록에 같은 기준을 적용해요.


![완성된 별도 예제: 온도 기록이에요. 시각 오름차순과 hall, room 열 순서를 함께 맞췄어요. 확인 문제의 거리 값과 달라요.](assets/02-example-sort.png)

완성된 별도 예제: 온도 기록이에요. 시각 오름차순과 hall, room 열 순서를 함께 맞췄어요. 확인 문제의 거리 값과 달라요.


**보충 예제 E1 · 필요한 경우만 실행**

argsort는 정렬된 값 자체 대신 **그 순서로 꺼낼 행 번호**를 돌려줘요. 같은 행 번호를 시각과 측정 배열에 적용해요.

이름 목록의 index(이름)는 그 이름의 위치 번호를 찾아요. 아래에서는 hall, room 이름으로 열 번호 목록을 만들고 [:, 열 번호 목록]으로 모든 행의 열을 그 순서로 꺼내요. 원하는 이름을 차례로 찾아 목록에 모으면 세 센서에도 적용할 수 있어요. for 반복문으로 모아도 괜찮아요.

이 예제의 °C는 섭씨 온도예요. room은 방, hall은 복도 센서예요. 정렬 전후 단위는 바뀌지 않아요.


In [ ]:
import numpy as np

example_time_ms = np.array([300, 100, 200])
# 원래 열은 room, hall 순서, 단위 °C예요.
example_temperature_c = np.array([[24, 21], [18, 20], [23, 25]])
example_sensor_names = ["room", "hall"]
example_row_order = np.argsort(example_time_ms, kind="stable")
example_sorted_c = example_temperature_c[example_row_order]
example_column_order = [
    example_sensor_names.index("hall"),
    example_sensor_names.index("room"),
]
example_named_c = example_sorted_c[:, example_column_order]  # hall, room 순서
print(example_time_ms[example_row_order])
print(example_named_c)

[100 200 300]
[[20 18]
 [25 23]
 [21 24]]


**조건 배열을 요약하는 방향**

- 마스크(mask)는 각 칸이 조건에 맞으면 True, 아니면 False인 배열이에요. bool은 참·거짓 자료형이며 물리 단위가 없어요.
- any(axis=1)은 **한 행의 열들 중 하나라도 True인지** 확인해요. 결과 길이는 기록 수예요.
- bool 배열의 sum(axis=0)은 **각 열에서 True가 나온 횟수**를 세어요. 결과 길이는 센서 수이고 단위는 회예요.

axis는 여기서 배열을 요약할 방향을 지정하는 인자예요. axis=1이면 열 방향을 모아 행별로, axis=0이면 행 방향을 모아 열별로 남겨요.


![별도 온도 예제: 각 행의 따뜻함 여부와 각 센서의 횟수는 서로 다른 요약이에요.](assets/03-example-mask.png)

별도 온도 예제: 각 행의 따뜻함 여부와 각 센서의 횟수는 서로 다른 요약이에요.


In [ ]:
# E2: E1 바로 다음에 실행해요. 온도가 22°C 이상인 칸을 표시해요.
example_warm_mask = example_named_c >= 22
print(example_warm_mask)
print(example_warm_mask.any(axis=1))  # 시각별로 하나라도 해당하는가?
print(example_warm_mask.sum(axis=0))  # hall, room 각각 몇 번인가?

[[False False]
 [ True  True]
 [False  True]]
[False  True  True]
[1 2]


**보충 예제 E2_SELECT · bool 마스크로 시각 고르기**

array[mask]는 bool 마스크가 True인 위치의 값만 고르는 표현이에요. 한 행씩 고르는 1차원 마스크의 길이는 선택할 배열의 행 수와 같아야 해요. 예를 들어 시각 3개를 고르면 마스크도 bool 3개여야 해요.

선택은 정렬이 아니에요. 선택된 값은 고르기 전 배열에서 나타난 순서를 유지해요. 아래에서는 먼저 시간순 배열을 만들었으므로 선택된 시각도 그 시간순을 유지해요. 2차원 거리 표에 행 마스크를 적용하면 해당 행과 그 행의 모든 열을 함께 골라요.

**실행 순서**

E1 → E2 → E2_SELECT. 온도 예제의 따뜻한 행에 해당하는 시각만 출력해요. 확인 문제의 거리 값이나 정답을 사용하지 않아요.


In [ ]:
# E2_SELECT: E1과 E2 바로 다음에 실행해요.
example_sorted_time_ms = example_time_ms[example_row_order]
example_warm_rows = example_warm_mask.any(axis=1)
print(example_sorted_time_ms[example_warm_rows])

[200 300]


## 준비 P0 공통 거리 기록


**사용할 입력 · P0_PREP 셀**

- recorded_time_ms: 받은 순서의 시각 배열, 형태 (6,), 단위 ms.
- recorded_distances_m: 같은 행 순서의 거리 배열, 형태 (6, 3), 단위 m.
- recorded_sensor_order: 거리 배열의 현재 열 이름. 이번 입력은 left, front, right 순서예요.
- danger_m: 위험 표시 기준 거리(m). 거리가 이 값 **이하**인 칸에 True를 표시해요.

분석 결과의 열 순서는 항상 front, left, right로 맞춰요. 센서별 횟수도 이 순서예요. 색은 앞쪽 파랑·왼쪽 초록·오른쪽 주황이며 이름도 함께 표시해요.


![확인 문제 Q1~Q4의 공통 입력. 시간 순서와 열 이름을 확인하세요. 위험 표시의 정답은 그리지 않았어요.](assets/04-recorded-input.png)

확인 문제 Q1~Q4의 공통 입력. 시간 순서와 열 이름을 확인하세요. 위험 표시의 정답은 그리지 않았어요.


In [ ]:
# P0_PREP: 새 커널에서는 이 준비 셀부터 실행해요.
import numpy as np

recorded_time_ms = np.array([600, 0, 400, 200, 800, 1000])
recorded_sensor_order = ["left", "front", "right"]
recorded_distances_m = np.array(
    [
        [0.60, 0.40, 0.65],  # 600ms
        [0.80, 1.20, 0.90],  #   0ms
        [0.70, 0.55, 0.30],  # 400ms
        [0.35, 0.90, 0.75],  # 200ms
        [0.50, 0.25, 0.45],  # 800ms
        [0.90, 0.80, 0.55],  # 1000ms
    ],
    dtype=float,
)
print("받은 열 이름:", recorded_sensor_order)
print("배열 형태:", recorded_distances_m.shape)

받은 열 이름: ['left', 'front', 'right']
배열 형태: (6, 3)


## 확인 문제 Q1 센서 기록 분석 함수

**필수 · 약 20분**

**조건**

시각은 서로 다른 0 이상의 정수이고, 거리 값은 유한한 0 이상의 수예요. 두 입력의 행 수는 같고 기록은 하나 이상이에요. 열 이름은 front, left, right가 한 번씩 들어 있어요. danger_m은 유한한 양수예요. 이 조건을 검사하는 예외 처리 기능은 Q1 범위가 아니에요.

**사용할 변수·함수와 실행 순서**

바로 앞 **준비 P0 / P0_PREP**의 recorded_time_ms, recorded_distances_m, recorded_sensor_order를 사용해요. 다른 교재의 코드가 필요하지 않아요. 새 커널은 P0_PREP → Q1_CODE → Q1_RUN 순서로 실행해요.

**할 일**

Q1_CODE의 analyze_sensor_log 함수 안에서 다음 처리를 연속으로 구현하세요.

1. 시간 오름차순 행 번호를 구해 시각과 거리 행을 함께 정리해요.
2. recorded_sensor_order에서 이름으로 열 번호를 찾아 front, left, right로 정리해요. 이번 입력의 번호를 상수로 고정하지 않아요.
3. 정리한 거리와 danger_m을 비교해 칸별 danger_mask를 만들어요. 같은 값도 위험 표시예요.
4. 한 센서라도 위험 표시인 행의 danger_rows와 그 시각 danger_times_ms를 만들어요.
5. 센서별 True 횟수 danger_count_by_sensor를 구해요.
6. 아래 Docstring의 **7개 키**를 가진 사전을 반환해요. 입력 배열·열 이름은 변경하지 않아요.

**답할 곳과 결과 형식**

Q1_CODE 함수의 raise 줄을 본인 코드로 바꾸고 Q1_RUN을 실행해요. Q1_ANSWER에는 열 이름과 횟수의 연결, 행별 표시와 센서별 횟수의 차이를 적어요. 반환 배열의 형태·단위는 함수 바로 아래 Docstring에 명시돼 있어요. 숫자 정답은 시도 후 검사해요.


In [ ]:
def analyze_sensor_log(
    recorded_time_ms, recorded_distances_m, recorded_sensor_order, danger_m
):
    """시간·센서 순서에 맞춰 거리와 위험 기록을 정리한다.

    매개변수:
        recorded_time_ms: 기록 시각 배열, 형태 (N,), 단위 ms.
        recorded_distances_m: 센서별 거리 배열, 형태 (N, 3), 단위 m.
        recorded_sensor_order: 입력 거리 배열의 열 이름 목록.
        danger_m: 위험으로 표시할 기준 거리, 단위 m.

    반환값:
        정렬된 시각·거리와 위험 여부·위험 시각·센서별 위험 횟수를
        담은 딕셔너리. 입력 배열과 목록은 변경하지 않는다.
    """
    # 1. 시각이 빠른 순서대로 행 번호를 구해요.
    time_order = np.argsort(recorded_time_ms)

    # 시각과 거리의 행에 같은 순서를 적용해 짝을 유지해요.
    time_ms = recorded_time_ms[time_order]
    distances_by_time = recorded_distances_m[time_order]

    # 2. 원하는 센서 순서에 맞는 기존 열 번호를 구해요.
    sensor_order = ["front", "left", "right"]
    column_order = [recorded_sensor_order.index(name) for name in sensor_order]
    distances_m = distances_by_time[:, column_order]

    # 3. 각 거리값이 기준 이하인지 비교해요.
    danger_mask = distances_m <= danger_m

    # 4. 각 행에 위험한 센서가 하나라도 있는지 확인해요.
    danger_rows = danger_mask.any(axis=1)

    # 위험한 행의 시각만 가져와요.
    danger_times_ms = time_ms[danger_rows]

    # 5. 센서별로 위험한 횟수를 세요. axis=0은 행 방향으로 세기예요.
    danger_count_by_sensor = danger_mask.sum(axis=0)

    return {
        "sensor_order": sensor_order,
        "time_ms": time_ms,
        "distances_m": distances_m,
        "danger_mask": danger_mask,
        "danger_rows": danger_rows,
        "danger_times_ms": danger_times_ms,
        "danger_count_by_sensor": danger_count_by_sensor,
    }

In [ ]:
# Q1_RUN: Q1_CODE가 완성된 후 실행해요.
time_before = recorded_time_ms.copy()
distances_before = recorded_distances_m.copy()
sensor_order_before = recorded_sensor_order.copy()
analysis_040 = analyze_sensor_log(
    recorded_time_ms, recorded_distances_m, recorded_sensor_order, danger_m=0.40
)
# analysis_040은 0.40m 기준의 분석 결과예요.
for name, value in analysis_040.items():
    print(name, "=", value)
assert np.array_equal(recorded_time_ms, time_before)
assert np.array_equal(recorded_distances_m, distances_before)
assert recorded_sensor_order == sensor_order_before

sensor_order = ['front', 'left', 'right']
time_ms = [   0  200  400  600  800 1000]
distances_m = [[1.2  0.8  0.9 ]
 [0.9  0.35 0.75]
 [0.55 0.7  0.3 ]
 [0.4  0.6  0.65]
 [0.25 0.5  0.45]
 [0.8  0.9  0.55]]
danger_mask = [[False False False]
 [False  True False]
 [False False  True]
 [ True False False]
 [ True False False]
 [False False False]]
danger_rows = [False  True  True  True  True False]
danger_times_ms = [200 400 600 800]
danger_count_by_sensor = [2 1 1]


**Q1_ANSWER · Q1 답안**

- 센서별 위험 횟수
  front 2회, left 1회, right 1회

- 두 결과의 차이
  danger_rows는 각 시각에 위험한 센서가 하나라도 있는지 나타낸다. danger_count_by_sensor는 각 센서가 전체 기록에서 위험했던 횟수를 나타낸다.

- 입력이 바뀌지 않았는지
  시각 배열, 거리 배열, 센서 이름 목록이 바뀌지 않았다


**보충 예제 E2_COMPARE · 기존 표시와 비교해 새 표시 찾기**

**쉬운 뜻**

비교 결과에서 True가 됐지만 기존 결과에서는 False였던 곳만 골라요.

**생활 비유**

점검표에 원래 있던 표시를 빼고 이번에 새로 붙은 표시만 세는 것과 같아요.

**실제 용어**

bool 배열의 차이 마스크예요. ~는 bool 배열의 True·False를 뒤집고, &는 같은 칸에서 두 조건이 함께 True인지 확인해요. 두 배열은 같은 행·열의 뜻과 같은 형태로 맞춰져 있어야 해요.

새 칸은 칸별 마스크끼리 비교해요. 새 행은 각 결과의 행별 표시끼리 비교해요. 이미 True인 칸이 있던 행에 다른 True 칸이 추가돼도 그 행 자체는 새 행이 아니에요.

아래 예제는 방·복도 온도이고 단위는 °C예요. 24°C 이상 표시와 22°C 이상 표시를 비교해요. 온도에서는 기준을 낮춰 표시가 늘어난 별도 예제이며, 확인 문제의 거리·기준값과 달라요.


![별도 온도 예제의 새 칸·새 행 비교](assets/07-example-added-mask.png)

별도 예제: T는 True, F는 False예요. 첫 행에는 새 칸이 생겼지만 행 자체는 이미 표시돼 있었어요.


In [ ]:
# E2_COMPARE: 다른 예제 셀에 의존하지 않는 완성된 별도 예제예요.
import numpy as np

example_values_c = np.array([[25, 23], [19, 23], [26, 20]])
example_old_mask = example_values_c >= 24
example_new_mask = example_values_c >= 22
example_added_cells = example_new_mask & ~example_old_mask
example_added_rows = example_new_mask.any(axis=1) & ~example_old_mask.any(axis=1)
print("새 칸:", example_added_cells)
print("새 행:", example_added_rows)
print("방·복도 추가 횟수(회):", example_added_cells.sum(axis=0))

새 칸: [[False  True]
 [False  True]
 [False False]]
새 행: [False  True False]
방·복도 추가 횟수(회): [0 2]


## 확인 문제 Q2 새 위험 표시 직접 비교하기

**필수 · 약 15분**

**조건**

같은 기록을 기준 거리 0.40m와 0.60m로 분석해요. 기준 이하인 값은 위험 표시예요. 두 결과는 같은 시간·센서 순서로 정리돼 있으므로 같은 위치끼리 비교할 수 있어요.

**사용할 변수·함수와 실행 순서**

- 공통 입력 3개: 준비 P0 / P0_PREP의 recorded_time_ms, recorded_distances_m, recorded_sensor_order.
- 분석 함수: Q1 / Q1_CODE의 analyze_sensor_log. 분석 전체를 두 조건으로 실행하는 것이 오늘 재사용의 목표예요.
- Q2_CODE 한 셀에 두 결과의 준비 코드, 본인이 작성할 부분, 출력 코드를 모았어요.
- 새 커널: P0_PREP → Q1_CODE → Q2_CODE. 새 비교 연산이 낯설면 바로 앞 E2_COMPARE를 먼저 실행해요. Q1_RUN·Q2의 이전 결과는 필요하지 않아요.

**할 일**

Q2_CODE의 raise 줄을 다음 다섯 변수를 만드는 코드로 바꾸세요. 작은 함수를 추가로 만들 필요는 없어요.

1. added_cell_mask: 0.60m 결과에서는 True이고 0.40m 결과에서는 False인 칸.
2. added_row_mask: 0.60m 결과에서는 위험 행이고 0.40m 결과에서는 위험 행이 아닌 행.
3. added_times_ms: added_row_mask가 가리키는 시간순 시각. bool 마스크로 시각을 골라요.
4. added_count_by_sensor: 새로 표시된 칸을 센서별로 센 추가 횟수.
5. boundary_mask: 제공된 boundary_distances_m의 각 값이 0.40m 이하인지 표시한 배열. 기준 바로 아래·같음·바로 위를 직접 비교해요.

**출력 계약**

- added_cell_mask: 형태 (6, 3), bool 배열, 시간순·front/left/right 열 순서, 단위 없음.
- added_row_mask: 형태 (6,), bool 배열, 시간순, 단위 없음.
- added_times_ms: 형태 (새 위험 행 수,), 정수 ndarray, 시간순, 단위 ms.
- added_count_by_sensor: 형태 (3,), 정수 ndarray, front/left/right 순서, 단위 회.
- boundary_mask: 형태 (3,), bool 배열, boundary_distances_m의 받은 순서, 단위 없음.

**답할 곳과 결과 형식**

Q2_CODE에 구현과 출력을 저장하고 Q2_ANSWER에는 새 칸과 새 행의 의미 차이, 센서별 증가 해석, 기준과 같은 값이 포함됐는지 확인한 근거만 짧게 적어요. 이미 출력한 목록을 다시 베껴 적지 않아요. 숫자 정답과 완성 코드는 시도 후 제공해요.


In [ ]:
# Q2_CODE: 두 결과 준비 → 본인 비교 코드 → 출력 순서예요.
analysis_040 = analyze_sensor_log(
    recorded_time_ms, recorded_distances_m, recorded_sensor_order, 0.40
)
analysis_060 = analyze_sensor_log(
    recorded_time_ms, recorded_distances_m, recorded_sensor_order, 0.60
)
assert np.array_equal(analysis_040["time_ms"], analysis_060["time_ms"])
assert analysis_040["sensor_order"] == analysis_060["sensor_order"]
boundary_distances_m = np.array([0.39, 0.40, 0.41])  # 앞쪽 거리, 단위 m

# added_cell_mask: 한 시각마다 센서 3개의 참·거짓을 담은 배열. 예: [[False, True, False], ...] → 총 6행 3열
# added_row_mask: 시각별 참·거짓 1개씩. 예: [True, False, ...] → 총 6개
# added_times_ms: 해당하는 시각들. 예: [200, 600] → 개수는 결과에 따라 달라져
# added_count_by_sensor: 센서 3개 각각의 횟수. 예: [1, 0, 2] → 총 3개
# boundary_mask: 센서 3개 각각의 참·거짓. 예: [False, True, False] → 총 3개

# 새 기준에서는 위험하지만, 예전 기준에서는 위험하지 않았던 칸
added_cell_mask = added_cell_mask = (analysis_060["danger_mask"]) & (
    ~analysis_040["danger_mask"]
)
# 0.60m 기준에서는 위험한 행이고, 0.40m 기준에서는 위험하지 않았던 행”
added_row_mask = analysis_060["danger_rows"] & ~analysis_040["danger_rows"]
# 새로 위험해진 시각
added_times_ms = analysis_040["time_ms"][added_row_mask]
added_count_by_sensor = added_cell_mask.sum(axis=0)
boundary_mask = boundary_distances_m <= 0.40

print("새 위험 칸:", added_cell_mask)
print("새 위험 행:", added_row_mask)
print("새 위험 시각(ms):", added_times_ms)
print("센서 순서:", analysis_040["sensor_order"])
print("센서별 추가 횟수(회):", added_count_by_sensor)
print("경계 입력(m):", boundary_distances_m)
print("0.40m 이하 표시:", boundary_mask)

새 위험 칸: [[False False False]
 [False False False]
 [ True False False]
 [False  True False]
 [False  True  True]
 [False False  True]]
새 위험 행: [False False False False False  True]
새 위험 시각(ms): [1000]
센서 순서: ['front', 'left', 'right']
센서별 추가 횟수(회): [1 2 2]
경계 입력(m): [0.39 0.4  0.41]
0.40m 이하 표시: [ True  True False]


**Q2_ANSWER · Q2 답안**

- 새 칸과 새 행을 각각 세는 이유와 두 결과의 차이
  - 새 칸은 새로 위험해진 개별 센서 측정값이고, 새 행은 이전에는 위험하지 않았지만 새 기준에서 처음 위험해진 시각이다. 이번에는 새 칸이 5개, 새 행은 1000ms의 1개다. 이미 위험했던 시각에 새 칸이 생겨도 새 행으로 세지 않는다.

- 센서별 추가 횟수로 해석한 변화
  - front는 1회, left는 2회, right는 2회 추가됐다. 위험 기준을 0.40m에서 0.60m로 넓히면서 센서별 전체 위험 횟수는 [2, 1, 1]에서 [3, 3, 3]으로 늘었다.

- 기준과 같은 값을 포함했는지 확인한 입력·출력 근거
  - [0.39, 0.40, 0.41]에 0.40m 이하 조건을 적용한 결과는 [True, True, False]다. 기준과 같은 0.40m도 True이므로 포함된다.


## 열의 값과 이름이 함께 움직여야 해요


**별도 상자 예제 E3**

상자 열을 옮겼는데 이름을 그대로 두면 컵 개수를 책 개수라고 읽게 돼요. 배열 형태가 맞아도 값의 뜻은 틀릴 수 있어요. 아래 예제의 숫자는 개수(개)이며 확인 문제의 거리 값과 달라요.


![별도 예제: 열 번호 [1, 2, 0]에 맞춰 값과 이름을 함께 옮겨요.](assets/05-example-column-labels.png)

별도 예제: 열 번호 [1, 2, 0]에 맞춰 값과 이름을 함께 옮겨요.


In [ ]:
# E3은 독립적인 별도 예제예요.
example_names = ["책", "컵", "펜"]
example_counts = [5, 8, 2]
example_move = [1, 2, 0]
print([example_names[index] for index in example_move])
print([example_counts[index] for index in example_move])

['컵', '펜', '책']
[8, 2, 5]


## 확인 문제 Q3 열 이름 오류 수정

**필수 · 약 10분**

**조건**

Q3_DEBUG는 P0 거리 열을 [2, 0, 1] 순서로 옮겼는데, permuted_sensor_order의 이름은 그 이동을 반영하지 못했어요. 행·시각·거리 값 자체는 새로 측정한 것이 아니에요. 고칠 대상은 이름 목록 한 줄이에요.

**사용할 변수·함수와 실행 순서**

P0_PREP의 공통 입력 3개와 Q1_CODE의 analyze_sensor_log를 사용해요. Q3_DEBUG는 기준 결과와 옮긴 결과를 바로 옆에서 함께 만들어요. 새 커널은 P0_PREP → Q1_CODE → Q3_DEBUG 순서예요. Q2 결과를 가져오지 않아요.

**할 일**

1. 받은 열 순서와 이동 번호를 보고 새 열의 실제 의미를 예상해 Q3_ANSWER에 적어요.
2. 고치기 전 Q3_DEBUG를 실행해 거리 배열 일치, 센서별 횟수, 행별 위험 표시를 비교해요. 수정 전 출력도 Q3_ANSWER에 남겨요.
3. permuted_sensor_order 한 줄만 고쳐 같은 셀을 다시 실행해요. 함수나 거리 값은 바꾸지 않아요.
4. 두 실행을 비교해 무엇이 복구됐는지 설명해요. 행별 위험 표시만으로 열 이름 오류를 찾을 수 있는지도 출력 근거로 판단해요.

**답할 곳과 결과 형식**

Q3_DEBUG의 수정한 한 줄·최신 출력과 Q3_ANSWER의 수정 전후 비교. 별도 A/B 이름이나 다른 반환 키를 만들지 않아요.


In [ ]:
# Q3_DEBUG: [2, 0, 1]은 새 열이 가져올 원래 열 번호예요.
permuted_distances_m = recorded_distances_m[:, [2, 0, 1]].copy()
# permuted_sensor_order = ["front", "left", "right"]  # 의도적으로 잘못된 한 줄
permuted_sensor_order = ["right", "left", "front"]

reference_analysis = analyze_sensor_log(
    recorded_time_ms, recorded_distances_m, recorded_sensor_order, 0.40
)
permuted_analysis = analyze_sensor_log(
    recorded_time_ms, permuted_distances_m, permuted_sensor_order, 0.40
)
print(
    "거리 배열 일치:",
    np.array_equal(reference_analysis["distances_m"], permuted_analysis["distances_m"]),
)
print("기준 센서별 횟수:", reference_analysis["danger_count_by_sensor"])
print("옮긴 센서별 횟수:", permuted_analysis["danger_count_by_sensor"])
print(
    "행별 표시 일치:",
    np.array_equal(reference_analysis["danger_rows"], permuted_analysis["danger_rows"]),
)

거리 배열 일치: True
기준 센서별 횟수: [2 1 1]
옮긴 센서별 횟수: [2 1 1]
행별 표시 일치: True


**Q3_ANSWER · Q3 답안**

- 옮긴 열의 실제 이름을 예상한 순서: ["right", "left", "front"]
- 수정 전 거리 배열·횟수·행별 표시 비교: 기준 [2 1 1], 옮긴 결과 [1 1 2]
- 고친 이름 목록 한 줄: permuted_sensor_order = ["right", "left", "front"]
- 수정 후 비교와 행별 표시만으로 충분한지 설명: 센서 이름을 [right, left, front]로 고친 뒤 다시 실행하니, 기준 결과와 옮긴 결과의 거리 배열과 센서별 위험 횟수가 같았다.

행별 위험 표시만으로는 충분하지 않다. 센서 열 이름이 잘못되어도 한 행에 위험한 센서가 하나라도 있는지는 같게 나올 수 있기 때문이다


## 빠진 값과 잘못된 측정을 구분하기


**쉬운 뜻**

기준과 비교하기 전에 측정값으로 쓸 수 있는 행인지 확인해요.

**생활 비유**

영수증 금액 하나가 비어 있으면 그 칸을 0원이라고 우겨 넣고 계산하지 않아요. 확인이 필요한 영수증을 따로 표시해요.

**실제 용어**

측정 품질 검사와 유효 행 마스크예요. NaN은 Not a Number의 줄임말로, 숫자로 나타내지 못하는 값을 표시해요. np.isfinite는 NaN이나 무한대가 아닌 유한한 값인지 칸마다 확인해요.


**오늘의 품질 규칙**

모든 센서 값이 유한하고 0 이상인 행만 분석해요. 유효하지 않은 행은 이번 분석에서 제외하고 시각을 따로 남겨요. 제외는 안전하다는 판정이 아니라 측정 확인이 필요하다는 뜻이에요.

- 배열 조건을 &로 연결할 때 각 조건을 괄호로 감싸요. &는 bool 배열에서 두 조건이 함께 True인지 확인해요.
- all(axis=1)은 한 행의 모든 센서가 True인지 확인해요. any(axis=1)과 역할이 달라요.
- bool 마스크 앞의 ~는 True와 False를 뒤집어요. 제외할 행을 고르는 데 쓸 수 있어요.


![완성된 별도 무게 예제: 단위 kg(킬로그램). 빈 값이나 음수 무게가 한 칸이라도 있으면 그 행은 유효하지 않아요.](assets/06-example-quality.png)

완성된 별도 무게 예제: 단위 kg(킬로그램). 빈 값이나 음수 무게가 한 칸이라도 있으면 그 행은 유효하지 않아요.


In [ ]:
# E4: 거리 문제와 별개인 저울 예제예요.
import numpy as np

example_weights_kg = np.array([[2.0, 4.0], [np.nan, 3.0], [-0.5, 2.0]])
example_valid_cells = np.isfinite(example_weights_kg) & (example_weights_kg >= 0)
example_valid_rows = example_valid_cells.all(axis=1)
print(example_valid_cells)
print(example_valid_rows)
print("제외 행 번호:", np.arange(3)[~example_valid_rows])

[[ True  True]
 [False  True]
 [False  True]]
[ True False False]
제외 행 번호: [1 2]


## 확인 문제 Q4 유효한 행만 함께 선택하기

**필수 · 약 15분**

**조건**

Q4_CODE에서 새 복사본 quality_distances_m의 [2, 1]을 NaN, [4, 2]를 -0.10m로 바꿔요. 첫 번호는 받은 배열의 행, 둘째 번호는 받은 배열의 열이에요. P0 원본은 바꾸지 않아요. 이번 자료에서는 제외 후에도 유효한 행이 남아요.

**사용할 변수·함수와 실행 순서**

P0_PREP의 공통 입력 3개와 Q1_CODE의 analyze_sensor_log를 사용해요. 새 커널은 P0_PREP → Q1_CODE → Q4_CODE 순서예요. Q2·Q3의 사전이나 수정 상태는 필요하지 않아요. Q1 함수에 품질 검사를 새로 섞지 않고 분석 전 행 선택만 여기서 추가해요.

**중간 결과 약속**

- valid_rows: 받은 행 순서의 bool ndarray, 형태 (6,), 단위 없음.
- clean_time_ms: 선택한 받은 순서의 1차원 정수 ndarray, 형태 (유효 행 수,), 단위 ms.
- clean_distances_m: clean_time_ms와 같은 선택 행 순서의 숫자 ndarray, 형태 (유효 행 수, 3), 단위 m. 열은 받은 센서 순서 그대로예요.
- removed_time_ms: 제외한 받은 순서의 1차원 정수 ndarray, 형태 (제외 행 수,), 단위 ms.

이 네 중간 결과를 만들 때 먼저 정렬하지 않아요. 뒤의 analyze_sensor_log가 시간·센서 순서 정리를 담당해요.

**할 일**

1. 모든 센서가 유한하고 0 이상인 행을 valid_rows에 표시해요. 형태는 (6,)인 bool 배열이에요.
2. 같은 valid_rows로 시각과 quality_distances_m을 함께 골라 clean_time_ms와 clean_distances_m을 만들어요. 시각만 따로 정렬하거나 거리만 걸러내지 않아요.
3. 제외한 시각을 removed_time_ms에 담아요.
4. 고른 자료를 같은 분석 함수에 넣어 기준 0.40m로 다시 실행해요.
5. 출력에서 제외 시각·남은 시각·위험 시각을 구분하고, 제외한 행을 안전하다고 보고하면 안 되는 이유를 설명해요.

**답할 곳과 결과 형식**

Q4_CODE의 raise 줄을 위 변수들을 만드는 코드로 바꾸고 실행해요. Q4_ANSWER에는 세 종류의 시각 목록(ms), 시각과 거리 행의 연결 근거, 제외의 의미를 적어요. 정답 함수나 완성 출력은 제공하지 않았어요.


In [ ]:
# Q4_CODE: P0의 원본 대신 품질 실험용 새 배열을 만들어요.
quality_distances_m = recorded_distances_m.copy()
quality_distances_m[2, 1] = np.nan
quality_distances_m[4, 2] = -0.10

# 각 행의 거리 3개가 모두 유한하고 0 이상인지 확인
valid_rows = (np.isfinite(quality_distances_m) & (quality_distances_m >= 0)).all(axis=1)

# 남길 행과 같은 위치의 시각·거리 선택
clean_time_ms = recorded_time_ms[valid_rows]
clean_distances_m = quality_distances_m[valid_rows]

# 제외한 행과 같은 위치의 시각 선택
removed_time_ms = recorded_time_ms[~valid_rows]

clean_analysis = analyze_sensor_log(
    clean_time_ms, clean_distances_m, recorded_sensor_order, danger_m=0.40
)
print("제외 시각(받은 순서, ms):", removed_time_ms)
print("남은 시각(시간순, ms):", clean_analysis["time_ms"])
print("위험 시각(ms):", clean_analysis["danger_times_ms"])
print("센서 순서:", clean_analysis["sensor_order"])
print("센서별 횟수(회):", clean_analysis["danger_count_by_sensor"])

제외 시각(받은 순서, ms): [400 800]
남은 시각(시간순, ms): [   0  200  600 1000]
위험 시각(ms): [200 600]
센서 순서: ['front', 'left', 'right']
센서별 횟수(회): [1 1 0]


**Q4_ANSWER · Q4 답안**

- 제외 시각(받은 순서, ms): [400 800]
- 남은 시각(시간순, ms): [ 0 200 600 1000]
- 위험 시각(ms): [200 600]
- 시각과 거리 행이 같은 기록을 가리키는 근거: valid_rows를 시각 배열과 거리 배열에 똑같이 적용해 같은 위치의 행을 함께 남겼다. 이후 시간순으로 정렬할 때도 시각과 거리 행을 같은 순서로 옮겼다.
- 제외한 행의 의미와 추가 확인이 필요한 이유: 제외한 행에는 NaN이나 음수처럼 유효하지 않은 거리값이 들어 있다. 분석에서 제외했다는 것은 안전하다는 뜻이 아니다. 실제 거리와 위험 여부를 알 수 없으므로 센서를 확인하거나 다시 측정해야 한다.


## 점검과 제출


**제출할 결과물**

- day24.ipynb에 Q1 함수, Q1~Q4 저장 출력과 짧은 답안을 남겨요.
- Q3은 수정 전 비교를 답안에 남기고 코드에는 고친 이름 목록을 저장해요.
- Q4는 품질 실험용 복사본만 바꿔요. P0 원본은 그대로 남겨요.

**이해 확인은 답안에 통합돼 있어요**

행·열 정리가 값의 뜻을 보존하는 이유, 위험 기준 변경의 영향, 열 이름 오류가 일부 요약에서 숨는 이유, 측정 제외와 안전 판정의 차이를 Q1~Q4에서 설명하면 충분해요. 같은 질문을 다시 제출할 필요는 없어요.


**막힐 때 확인할 순서**

1. NameError: 해당 문제의 실행 순서에서 P0_PREP와 Q1_CODE를 먼저 실행했는지 확인해요.
2. 열별 결과가 이상함: 숫자 위치를 고정하지 않고 recorded_sensor_order에서 이름으로 찾았는지 확인해요.
3. 행별 결과 길이가 3임: 기록 수와 센서 수를 구분하고 집계 방향을 확인해요.
4. 거리는 걸렀는데 시각이 어긋남: 같은 valid_rows를 두 입력에 적용했는지 확인해요.
5. np.array_equal이 False임: Q3에서는 이름 오류가 원인인지 비교해요. Q4의 NaN 자체를 같음 비교로 검사할 필요는 없어요.


**시간과 다음 진도**

실제 소요시간과 막힌 부분만 짧게 알려주세요. 막힌 곳이 없으면 회고 칸이 비어 있어도 괜찮아요. 60분을 넘으면 남은 부분부터 이어가고 새 필수 과제를 쌓지 않아요. 실제 풀이와 이해도를 보고 다음 분량을 조정해요.

**오늘 결과의 범위**

위험 표시는 기록을 분석하기 위한 기준이에요. 실제 로봇의 충돌 방지나 인증된 안전 거리라는 뜻은 아니에요. 이번에는 가상 거리 기록만 사용하며 새 설치·유료 장비가 필요 없어요. 자료 도착은 기상이나 즉시 공부 요청이 아니에요.


**자료 안내와 출처**

노트북 하나에 개념·그림·예제·문제·답안 칸·선택 타이머가 있어요. PDF는 같은 내용의 읽기 사본이며 타이머 사용 안내를 포함해요. 정답·해설은 시도 후 제공해요. 개인 맞춤 자료이며 KANT 공식 교재가 아니에요. 종일 부트캠프 주차를 개인 기간으로 환산하지 않아요.

기준 과정: https://kant-ai-campus.com/ai-physical

NumPy 공식 문서: https://numpy.org/doc/stable/reference/generated/numpy.argsort.html

행별 조건 확인: https://numpy.org/doc/stable/reference/generated/numpy.any.html

유효한 값과 행 검사: https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html · https://numpy.org/doc/stable/reference/generated/numpy.all.html

bool 마스크 선택: https://numpy.org/doc/stable/user/basics.indexing.html#boolean-array-indexing

bool 배열의 &와 ~: https://numpy.org/doc/stable/reference/generated/numpy.bitwise_and.html · https://numpy.org/doc/stable/reference/generated/numpy.invert.html

2026-11-12 교재. 기존 Python·NumPy 실행 환경을 사용해요.


## 선택 새로 측정하기

정말 다시 잴 때만 노트북의 아래 셀에서 enabled=False를 True로 바꾸고 초기화를 입력하세요. 기존 기록은 검증된 백업으로 보존한 뒤 시작 셀을 다시 실행해요.


In [ ]:
import stat as _stat
from pathlib import Path as _Path


def _load_shared_timer():
    # 교재 ID·날짜·노트북 이름은 제작 때 고정한 값이에요.
    _lesson_id = "day24-sensor-log-analysis"
    _lesson_date = "2026-11-12"
    _notebook_name = "day24.ipynb"

    def _safe(path, directory=False):
        if ".." in path.parts:
            raise RuntimeError("상위 폴더 경로는 사용하지 않아요.")
        for item in [*reversed(path.parents), path]:
            info = item.lstat()
            if (
                _stat.S_ISLNK(info.st_mode)
                or getattr(info, "st_file_attributes", 0) & 0x400
            ):
                raise RuntimeError("바로가기·junction·reparse 경로는 사용하지 않아요.")
        if directory and not _stat.S_ISDIR(info.st_mode):
            raise RuntimeError("예상한 폴더가 아니에요.")
        if not directory and not _stat.S_ISREG(info.st_mode):
            raise RuntimeError("예상한 일반 파일이 아니에요.")
        return path

    _cwd = _safe(_Path.cwd(), directory=True)
    _candidates = []
    for _folder in (_cwd, _cwd / _lesson_id):
        if _folder.name == _lesson_id and (_folder / _notebook_name).is_file():
            _safe(_folder, directory=True)
            _candidates.append(_safe(_folder / _notebook_name))
    if len(_candidates) != 1:
        raise RuntimeError("교재 폴더가 없거나 모호해요.")
    _notebook_path = _candidates[0]
    _shared_dir = _safe(_notebook_path.parent.parent / "_shared", directory=True)
    _helper = _safe(_shared_dir / "study_timer.py")
    # 검사한 파일을 메모리에서 로드해 __pycache__를 만들지 않아요.
    _before = _helper.lstat()
    _source = _helper.read_bytes()
    _after = _safe(_helper).lstat()
    _identity = lambda info: (info.st_dev, info.st_ino, info.st_mtime_ns, info.st_size)
    if _identity(_before) != _identity(_after):
        raise RuntimeError("읽는 동안 공통 도구 파일이 바뀌었어요.")
    _module = {"__name__": "physical_ai_shared_timer"}
    exec(compile(_source, str(_helper), "exec"), _module)
    return _module["create_timer"](
        _lesson_id, _lesson_date, _notebook_name, notebook_path=_notebook_path
    )


try:
    _study_timer = _load_shared_timer()
except (OSError, RuntimeError, SyntaxError):
    print(
        "타이머 경로를 확인하지 못했어요. 루트 또는 교재 폴더에서 열고 _shared 폴더를 함께 보관해 주세요. 기록은 바꾸지 않았어요."
    )
else:
    enabled = False  # 정말 다시 잴 때만 True로 바꾸세요.
    if enabled:
        _study_timer.action("reset", input("초기화하려면 초기화 입력: "), enabled=True)
    else:
        print("초기화는 비활성화되어 있어요.")

## 선택 학습 종료

마지막 문제까지 저장할 준비가 되면 아래 셀에서 직접 종료를 입력하세요. 빈 입력은 취소이고 기록을 바꾸지 않아요. 모두 실행 중에는 종료를 입력하지 마세요. 종료 결과 확인 → Ctrl+S 저장 → 이 대화에 다 풀었어라고 알려주기 순서예요.


In [ ]:
import stat as _stat
from pathlib import Path as _Path


def _load_shared_timer():
    # 교재 ID·날짜·노트북 이름은 제작 때 고정한 값이에요.
    _lesson_id = "day24-sensor-log-analysis"
    _lesson_date = "2026-11-12"
    _notebook_name = "day24.ipynb"

    def _safe(path, directory=False):
        if ".." in path.parts:
            raise RuntimeError("상위 폴더 경로는 사용하지 않아요.")
        for item in [*reversed(path.parents), path]:
            info = item.lstat()
            if (
                _stat.S_ISLNK(info.st_mode)
                or getattr(info, "st_file_attributes", 0) & 0x400
            ):
                raise RuntimeError("바로가기·junction·reparse 경로는 사용하지 않아요.")
        if directory and not _stat.S_ISDIR(info.st_mode):
            raise RuntimeError("예상한 폴더가 아니에요.")
        if not directory and not _stat.S_ISREG(info.st_mode):
            raise RuntimeError("예상한 일반 파일이 아니에요.")
        return path

    _cwd = _safe(_Path.cwd(), directory=True)
    _candidates = []
    for _folder in (_cwd, _cwd / _lesson_id):
        if _folder.name == _lesson_id and (_folder / _notebook_name).is_file():
            _safe(_folder, directory=True)
            _candidates.append(_safe(_folder / _notebook_name))
    if len(_candidates) != 1:
        raise RuntimeError("교재 폴더가 없거나 모호해요.")
    _notebook_path = _candidates[0]
    _shared_dir = _safe(_notebook_path.parent.parent / "_shared", directory=True)
    _helper = _safe(_shared_dir / "study_timer.py")
    # 검사한 파일을 메모리에서 로드해 __pycache__를 만들지 않아요.
    _before = _helper.lstat()
    _source = _helper.read_bytes()
    _after = _safe(_helper).lstat()
    _identity = lambda info: (info.st_dev, info.st_ino, info.st_mtime_ns, info.st_size)
    if _identity(_before) != _identity(_after):
        raise RuntimeError("읽는 동안 공통 도구 파일이 바뀌었어요.")
    _module = {"__name__": "physical_ai_shared_timer"}
    exec(compile(_source, str(_helper), "exec"), _module)
    return _module["create_timer"](
        _lesson_id, _lesson_date, _notebook_name, notebook_path=_notebook_path
    )


try:
    _study_timer = _load_shared_timer()
except (OSError, RuntimeError, SyntaxError):
    print(
        "타이머 경로를 확인하지 못했어요. 루트 또는 교재 폴더에서 열고 _shared 폴더를 함께 보관해 주세요. 기록은 바꾸지 않았어요."
    )
else:
    _study_timer.finish_interactive()